# Zadanie Dodatkowe: Cyfrowy Fantom Czaszki i Detekcja Krwotoku

To zadanie symuluje rzeczywisty problem kliniczny. Zamiast gotowego obrazu ze skanera, wygenerujemy **cyfrowy fantom głowy** w postaci macierzy surowych danych, do której dodamy szum kwantowy (charakterystyczny dla promieniowania rentgenowskiego).

Twoim zadaniem jako fizyka medycznego będzie przeliczenie tego sygnału na skalę Hounsfielda, a następnie odpowiednie zmanipulowanie parametrami okna (Windowing), aby uwidocznić ukrytą patologię (krwiak podtwardówkowy), który w standardowych ustawieniach jest niemal niewidoczny.

In [2]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 8)

## Krok 1: Wygenerowanie surowych danych z detektora (Symulacja)
Uruchom poniższą komórkę. Wygeneruje ona macierz `raw_detector_signal`, która udaje surowe dane zapisane w pliku DICOM (przed konwersją do HU). W nagłówku tego hipotetycznego pliku DICOM zapisano następujące parametry kalibracyjne:
- `Rescale Slope (m) = 1.0`
- `Rescale Intercept (b) = -1024`

In [5]:
# --- KOD GENERUJĄCY FANTOM (Nie musisz go modyfikować) ---
size = 256
x, y = np.meshgrid(np.linspace(-1, 1, size), np.linspace(-1, 1, size))
radius = np.sqrt(x**2 + y**2)

# Inicjalizacja pustej matrycy (Powietrze: -1000 HU)
phantom_hu = np.full((size, size), -1000.0)

# Kość czaszki (promień 0.8 do 0.85, HU: +1500)
phantom_hu[(radius < 0.85) & (radius >= 0.8)] = 1500.0

# Tkanka mózgowa (promień < 0.8, HU: +35)
phantom_hu[radius < 0.8] = 35.0

# Komory mózgu (Płyn MR, HU: +10)
phantom_hu[((x-0.15)**2 + (y-0.2)**2 < 0.02)] = 10.0
phantom_hu[((x+0.15)**2 + (y-0.2)**2 < 0.02)] = 10.0

# UKRYTA PATOLOGIA: Krwiak (Świeża krew, HU: +75)
phantom_hu[((x-0.6)**2 + (y+0.4)**2 < 0.015)] = 75.0

# Dodanie fizycznego szumu kwantowego (Gaussian noise)
np.random.seed(42)
noise = np.random.normal(loc=0, scale=15.0, size=(size, size))
phantom_hu += noise

# Konwersja odwrotna do "surowego" sygnału sprzętowego (Raw = HU - Intercept)
raw_detector_signal = (phantom_hu - (-1024)).astype(np.int16)

print("Fantom wygenerowany pomyślnie!")
print(f"Wymiary macierzy: {raw_detector_signal.shape}")
print(f"Min / Max surowego sygnału: {np.min(raw_detector_signal)} / {np.max(raw_detector_signal)}")

Fantom wygenerowany pomyślnie!
Wymiary macierzy: (256, 256)
Min / Max surowego sygnału: -34 / 2570


--- 
## Zadanie 1: Konwersja do skali Hounsfielda
Używając zmiennej `raw_detector_signal` oraz parametrów `m = 1.0` i `b = -1024`, przelicz surowy sygnał sprzętowy na obiektywną macierz gęstości radiologicznej `image_hu`.
Wydrukuj na ekran nową wartość minimalną i maksymalną. Sprawdź, czy minimum odpowiada w przybliżeniu wartości dla powietrza (ok. -1000 HU).

In [ ]:
# Miejsce na Twój kod (Zadanie 1):



## Zadanie 2: Funkcja Okienkowania
Skopiuj lub napisz ponownie funkcję `apply_window(image, window_width, window_level)`. Będzie Ci ona potrzebna w kolejnym kroku.

In [ ]:
# Miejsce na Twój kod (Zadanie 2):
def apply_window(image, window_width, window_level):
    # Zaimplementuj wzór obcinania i skalowania do 0-255
    pass

## Zadanie 3: Poszukiwanie patologii (Wizualizacja)
Wygenerowany fantom posiada ukryty "krwiak" (świeżą krew), której gęstość waha się w granicach 70-80 HU. Zdrowa tkanka mózgowa to około 30-40 HU. Przez obecność szumu kwantowego różnica ta jest trudna do uchwycenia.

Wyświetl obok siebie (używając `plt.subplots`) dwa obrazy:
1. **Klasyczne okno tkanki miękkiej** (WW = 400, WL = 40). Oceń, czy krwiak jest wyraźnie widoczny.
2. **Specjalistyczne okno mózgowe** (WW = 80, WL = 40). To okno jest bardzo wąskie, więc maksymalnie uwydatnia różnice w okolicach 40 HU.

*Wskazówka: Zastosuj `cmap='gray'`.*

In [ ]:
# Miejsce na Twój kod (Zadanie 3):



## Pytanie kontrolne na koniec:
Dlaczego w oknie tkanki miękkiej krwotok zlewa się z mózgiem, a w oknie mózgowym staje się bardzo jasny, mimo że w obu przypadkach parametr WL (środek okna) wynosił 40 HU? Jakie fizyczne znaczenie miało zmniejszenie WW z 400 na 80?